# Plotly + Dash tutorial


### Plotly

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import warnings

warnings.filterwarnings("ignore")

RANDOM_STATE = 1234

#### Chekhov's Gun

In [ ]:
# pio.renderers.default = "notebook"
pio.renderers.default

## Import and transform dataset

In [ ]:
import openml


ds = openml.datasets.get_dataset(39)
df, _, _, _ = ds.get_data()

In [ ]:
df.head()

### Bar plot

In [ ]:
#Option 1 - plotting using plotly.express

fig1 = px.bar(
    x=df["class"].unique(),
    y=df["class"].value_counts(),
    title="Class count",
    labels={"x": "class", "y": "count"}
)

fig1

In [ ]:
# Option 2 - plotting using plotly.graph_objects

class_counts = df["class"].value_counts()

fig2 = go.Figure(
    data=[
        go.Bar(
            x=class_counts.index,
            y=class_counts.values,
            name="count"
        )
    ]
)

fig2.update_layout(
    title="Class count",
    xaxis_title="class",
    yaxis_title="count",
    showlegend=False,
    template="plotly_white"
)

fig2

In [ ]:
# Option 3 - changing pandas backend to plotly 
pd.options.plotting.backend = "plotly"

fig3 = df["class"].value_counts().plot(kind="bar")
fig3.update_layout(showlegend=False)
fig3.update_layout(title="Class count")

fig3

In [ ]:
# For educational purposes, we will train our classificator using only 4 most common classes

df = df[df["class"].isin(["cp", "im", "pp", "imU"])]

### Exploring data

#### Scatter plot

In [ ]:
fig = px.scatter(
    df,
    x="mcg",
    y="gvh",
    color="class",
    symbol="class",
    title="Scatter plot: gvh vs mcg",
    hover_data=["lip", "chg", "aac", "alm1", "alm2"]
)

fig

Let's make it interactive!

In [ ]:
axis_cols = df.select_dtypes(include="number").columns.tolist()

default_x = "mcg" if "mcg" in axis_cols else axis_cols[0]
default_y = "gvh" if "gvh" in axis_cols else axis_cols[1]

fig = px.scatter(
    df,
    x=default_x,
    y=default_y,
    color="class",
    symbol="class",
    hover_data=["lip", "chg", "aac", "alm1", "alm2"],
    title=f"Scatter plot: {default_y} vs {default_x}",
    template="plotly_white"
)

trace_classes = [tr.name for tr in fig.data]

x_buttons = [
    dict(
        label=f"X: {col}",
        method="update",
        args=[
            {"x": [df.loc[df["class"] == cls, col] for cls in trace_classes]},
            {"xaxis": {"title": col}, "title": f"Scatter plot: {default_y} vs {col}"},
        ],
    )
    for col in axis_cols
]

y_buttons = [
    dict(
        label=f"Y: {col}",
        method="update",
        args=[
            {"y": [df.loc[df["class"] == cls, col] for cls in trace_classes]},
            {"yaxis": {"title": col}, "title": f"Scatter plot: {col} vs {default_x}"},
        ],
    )
    for col in axis_cols
]

fig.update_layout(
    template="plotly_white",
    margin=dict(t=120, r=30, l=60, b=60),
    updatemenus=[
        dict(
            type="dropdown",
            buttons=x_buttons,
            direction="down",
            x=0.68,
            y=1.10,
            xanchor="left",
            yanchor="top",
            showactive=True,
            active=axis_cols.index(default_x),
            pad=dict(r=8, t=4),
        ),
        dict(
            type="dropdown",
            buttons=y_buttons,
            direction="down",
            x=0.86,
            y=1.10,
            xanchor="left",
            yanchor="top",
            showactive=True,
            active=axis_cols.index(default_y),
            pad=dict(r=8, t=4),
        ),
    ],
)

fig.update_xaxes(title_text=default_x)
fig.update_yaxes(title_text=default_y)

fig

Let's create interactive histogram plot

In [ ]:
hist_cols = df.select_dtypes(include="number").columns.tolist()
default_col = "mcg" if "mcg" in hist_cols else hist_cols[0]

fig_hist_px = px.histogram(
    df,
    x=default_col,
    color="class",
    barmode="overlay",
    nbins=30,
    opacity=0.6,
    title=f"Interactive Histogram of {default_col}",
    template="plotly_white",
)

trace_classes = [tr.name for tr in fig_hist_px.data]

hist_buttons = [
    dict(
        label=col,
        method="update",
        args=[
            {"x": [df.loc[df["class"] == cls, col] for cls in trace_classes]},
            {
                "title": f"Interactive Histogram of {col}",
                "xaxis": {"title": col},
                "yaxis": {"title": "Count"},
            },
        ],
    )
    for col in hist_cols
]

fig_hist_px.update_layout(
    bargap=0.05,
    xaxis_title=default_col,
    yaxis_title="Count",
    updatemenus=[
        dict(
            type="dropdown",
            buttons=hist_buttons,
            x=0.75,
            y=1.16,
            xanchor="left",
            yanchor="top",
            showactive=True,
            active=hist_cols.index(default_col),
        )
    ],
    margin=dict(t=110, r=30, l=60, b=60),
)

fig_hist_px.show()

In [ ]:
# Drop lip and chg columns
df = df.drop(columns=['lip', 'chg'])

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

X = df.drop(columns=["class"]).copy()
y = df["class"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=RANDOM_STATE,
    shuffle=True,
    stratify=y,
)

rfc = RandomForestClassifier(
    n_estimators=20,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    max_features=2,
)

rfc.fit(X_train, y_train)
y_pred = rfc.predict(X_test)

print(classification_report(y_test, y_pred))

Visualize confusion matrix

In [ ]:
from sklearn.metrics import confusion_matrix
import plotly.figure_factory as ff

# Get class labels
class_labels = sorted(y_test.unique())

# Compute confusion matrix
cm = confusion_matrix(y_test, y_pred, labels=class_labels)

# Create annotated heatmap
fig_cm = ff.create_annotated_heatmap(
    z=cm,
    x=class_labels,
    y=class_labels,
    colorscale="Blues",
    showscale=True,
    annotation_text=cm,
)

# Update layout
fig_cm.update_layout(
    title="Confusion Matrix",
    xaxis_title="Predicted",
    yaxis_title="True",
    template="plotly_white",
    width=600,
    height=500,
    font=dict(size=12),
)

fig_cm.show()

Let's make interactive plot with slider to show how each tree classifies

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

if "rfc" not in globals() or not hasattr(rfc, "estimators_"):
    raise ValueError("Run the Random Forest training cell first.")

numeric_cols = X_train.select_dtypes(include="number").columns.tolist()
if len(numeric_cols) < 2:
    raise ValueError("Need at least 2 numeric features to plot tree boundaries.")

classes = list(rfc.classes_)
class_to_idx = {c: i for i, c in enumerate(classes)}

# Keep class colors stable across points and decision regions.
marker_palette = px.colors.qualitative.Bold
class_color_map = {cls: marker_palette[i % len(marker_palette)] for i, cls in enumerate(classes)}

# Use the same class colors for boundary planes to avoid color/label mismatch.
boundary_class_colors = [class_color_map[cls] for cls in classes]


def build_discrete_colorscale(colors):
    n = len(colors)
    if n == 1:
        return [[0.0, colors[0]], [1.0, colors[0]]]

    colorscale = []
    for i, color in enumerate(colors):
        left = i / n
        right = (i + 1) / n
        colorscale.append([left, color])
        colorscale.append([right, color])
    return colorscale


boundary_colorscale = build_discrete_colorscale(boundary_class_colors)


def to_class_index(pred):
    if pred in class_to_idx:
        return class_to_idx[pred]
    idx = int(pred)
    if 0 <= idx < len(classes):
        return idx
    raise ValueError(f"Cannot map prediction {pred!r} to class index")


def to_class_label(pred):
    return classes[to_class_index(pred)]


def pick_tree_feature_pair(tree, fallback_cols):
    importances = pd.Series(tree.feature_importances_, index=X_train.columns)
    ranked = [c for c in importances.sort_values(ascending=False).index if c in fallback_cols]

    pair = []
    for col in ranked:
        if col not in pair:
            pair.append(col)
        if len(pair) == 2:
            break

    if len(pair) < 2:
        for col in fallback_cols:
            if col not in pair:
                pair.append(col)
            if len(pair) == 2:
                break

    return pair[0], pair[1]


max_trees_to_show = min(30, len(rfc.estimators_))

# Precompute contour grids and train points for each tree using tree-specific feature pairs
tree_payload = []
for i, tree in enumerate(rfc.estimators_[:max_trees_to_show]):
    feature_1, feature_2 = pick_tree_feature_pair(tree, numeric_cols)

    step = 0.02
    x_min, x_max = X_train[feature_1].min() - 0.05, X_train[feature_1].max() + 0.05
    y_min, y_max = X_train[feature_2].min() - 0.05, X_train[feature_2].max() + 0.05
    xx, yy = np.meshgrid(np.arange(x_min, x_max, step), np.arange(y_min, y_max, step))

    base_row = X_train.median(numeric_only=True)
    grid_full = pd.DataFrame(np.tile(base_row.values, (xx.size, 1)), columns=X_train.columns)
    grid_full[feature_1] = xx.ravel()
    grid_full[feature_2] = yy.ravel()

    tree_pred = tree.predict(grid_full)
    z = np.array([to_class_index(p) for p in tree_pred], dtype=int).reshape(xx.shape)
    z_class = np.array(classes, dtype=object)[z]

    # Predict points on the same 2D plane used for boundary coloring.
    slice_points = pd.DataFrame(np.tile(base_row.values, (len(X_train), 1)), columns=X_train.columns)
    slice_points[feature_1] = X_train[feature_1].values
    slice_points[feature_2] = X_train[feature_2].values
    train_pred = np.array([to_class_label(p) for p in tree.predict(slice_points)], dtype=object)

    class_points = []
    for cls in classes:
        mask = y_train == cls
        class_points.append(
            {
                "x": X_train.loc[mask, feature_1],
                "y": X_train.loc[mask, feature_2],
                "pred": train_pred[mask],
            }
        )

    tree_payload.append(
        {
            "f1": feature_1,
            "f2": feature_2,
            "xgrid": xx[0],
            "ygrid": yy[:, 0],
            "z": z,
            "z_class": z_class,
            "class_points": class_points,
            "xrange": [float(x_min), float(x_max)],
            "yrange": [float(y_min), float(y_max)],
        }
    )

fig_tree_boundaries = go.Figure()
first = tree_payload[0]

# Decision plane for first tree (heatmap avoids fill overlap artifacts).
fig_tree_boundaries.add_trace(
    go.Heatmap(
        x=first["xgrid"],
        y=first["ygrid"],
        z=first["z"],
        hoverinfo="skip",
        zmin=-0.5,
        zmax=len(classes) - 0.5,
        zsmooth=False,
        colorscale=boundary_colorscale,
        opacity=0.55,
        showscale=False,
        name="Decision plane",
    )
)

# Training points overlay for first tree feature pair
for cls, pts in zip(classes, first["class_points"]):
    fig_tree_boundaries.add_trace(
        go.Scatter(
            x=pts["x"],
            y=pts["y"],
            customdata=np.array(pts["pred"], dtype=object),
            mode="markers",
            name=f"{cls}",
            marker=dict(
                size=8,
                color=class_color_map[cls],
                opacity=0.95,
                line=dict(width=0.8, color="white"),
            ),
            hovertemplate=(
                f"True class: {cls}<br>"
                "Plane predicts: %{customdata}<br>"
                f"{first['f1']}: %{{x:.3f}}<br>"
                f"{first['f2']}: %{{y:.3f}}<extra></extra>"
            ),
        )
    )

# Build frames so axes and points can change per tree
frames = []
for i, payload in enumerate(tree_payload):
    frame_data = [
        go.Heatmap(
            x=payload["xgrid"],
            y=payload["ygrid"],
            z=payload["z"],
            hoverinfo="skip",
            zmin=-0.5,
            zmax=len(classes) - 0.5,
            zsmooth=False,
            colorscale=boundary_colorscale,
            opacity=0.55,
            showscale=False,
        )
    ]

    for cls, pts in zip(classes, payload["class_points"]):
        frame_data.append(
            go.Scatter(
                x=pts["x"],
                y=pts["y"],
                customdata=np.array(pts["pred"], dtype=object),
                hovertemplate=(
                    f"True class: {cls}<br>"
                    "Plane predicts: %{customdata}<br>"
                    f"{payload['f1']}: %{{x:.3f}}<br>"
                    f"{payload['f2']}: %{{y:.3f}}<extra></extra>"
                ),
            )
        )

    frames.append(
        go.Frame(
            name=str(i),
            data=frame_data,
            traces=list(range(len(classes) + 1)),
            layout=go.Layout(
                title=(
                    f"Decision boundary of tree {i+1}/{max_trees_to_show} on {payload['f1']}/{payload['f2']}"
                    "<br><sup>Non-plotted features fixed to training medians</sup>"
                ),
                xaxis=dict(title=payload["f1"], range=payload["xrange"]),
                yaxis=dict(title=payload["f2"], range=payload["yrange"]),
            ),
        )
    )

fig_tree_boundaries.frames = frames

steps = []
for i in range(max_trees_to_show):
    steps.append(
        dict(
            method="animate",
            args=[[str(i)], {"mode": "immediate", "frame": {"duration": 0, "redraw": True}, "transition": {"duration": 0}}],
            label=str(i + 1),
        )
    )

fig_tree_boundaries.update_layout(
    title=(
        f"Decision boundary of tree 1/{max_trees_to_show} on {first['f1']}/{first['f2']}"
        "<br><sup>Non-plotted features fixed to training medians</sup>"
    ),
    xaxis=dict(title=first["f1"], range=first["xrange"]),
    yaxis=dict(title=first["f2"], range=first["yrange"]),
    template="plotly_white",
    width=980,
    height=680,
    sliders=[
        dict(
            active=0,
            currentvalue={"prefix": "Tree index: "},
            pad={"t": 35},
            steps=steps,
        )
    ],
)

fig_tree_boundaries.show()

And it's time to save the notebook to HTML with all our plots. Nothing can go wrong!